In [3]:
import pandas as pd
import numpy as np
from sklearn.linear_model import Lasso
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# =========================================================================
# 1. LOAD AND PREPARE DATASET
# =========================================================================
CSV_PATH = r"C:\Project\data\processed\flower_prices_with_rel_day.csv"
df = pd.read_csv(CSV_PATH)
df['DATE'] = pd.to_datetime(df['DATE'])
df = df.sort_values('DATE').reset_index(drop=True)

# Create lag and rolling features (backfilled to maintain full row count)
for lag in [1, 2, 3, 7]:
    df[f'price_lag_{lag}'] = df['price_updated'].shift(lag).bfill()

df['rolling_mean_3d'] = df['price_updated'].shift(1).rolling(3).mean().bfill()
df['rolling_mean_7d'] = df['price_updated'].shift(1).rolling(7).mean().bfill()

# Fill dummy columns with 0 where NaN
dummy_cols = [f'rel_day_{i}' if i < 0 else (f'rel_day_p{i}' if i > 0 else 'rel_day_0') for i in range(-7, 8)] + ['in_festival_window']
for col in dummy_cols:
    df[col] = df[col].fillna(0)

feature_cols = [
    'price_lag_1', 'price_lag_2', 'price_lag_3', 'price_lag_7',
    'rolling_mean_3d', 'rolling_mean_7d',
    'is_festival'
] + dummy_cols

X = df[feature_cols]
y = df['price_updated']

# =========================================================================
# 2. CHRONOLOGICAL SPLIT & SCALING
# =========================================================================
total_rows = len(df)
train_end = int(total_rows * 0.70)
val_end = int(total_rows * 0.85)

X_train, y_train = X.iloc[:train_end], y.iloc[:train_end]
X_test, y_test = X.iloc[val_end:], y.iloc[val_end:]

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# =========================================================================
# 3. TRAIN LASSO REGRESSION MODEL
# =========================================================================
lasso = Lasso(alpha=1.0, max_iter=10000)
lasso.fit(X_train_scaled, y_train)

# =========================================================================
# 4. EVALUATE MODEL ACCURACY
# =========================================================================
y_pred = lasso.predict(X_test_scaled)

r2 = r2_score(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae = mean_absolute_error(y_test, y_pred)

print("="*50)
print("MODEL ACCURACY EVALUATION (TEST SET)")
print("="*50)
print(f"R² Score Accuracy : {r2 * 100:.2f}% (Variance explained)")
print(f"RMSE (Root Mean Sq) : ₹{rmse:.2f} (Average error magnitude)")
print(f"MAE (Mean Absolute) : ₹{mae:.2f} (Average absolute miss)")

# =========================================================================
# 5. RANDOM DATE PREDICTION FUNCTION
# =========================================================================
def predict_flower_price_for_date(target_date_str, threshold_price=600):
    """
    Checks if a given date exists in the dataset and predicts whether 
    it will experience a price spike based on festival windows and price momentum.
    """
    target_date = pd.to_datetime(target_date_str)
    match = df[df['DATE'] == target_date]
    
    if match.empty:
        return f"Date {target_date_str} not found in dataset range ({df['DATE'].min().date()} to {df['DATE'].max().date()})."
    
    row_idx = match.index[0]
    features_vector = X.iloc[[row_idx]]
    scaled_vector = scaler.transform(features_vector)
    
    predicted_price = lasso.predict(scaled_vector)[0]
    actual_price = y.iloc[row_idx]
    is_fest = df.loc[row_idx, 'is_festival']
    rel_d = df.loc[row_idx, 'rel_day']
    fest_name = df.loc[row_idx, 'festival_name']
    
    print("\n" + "="*50)
    print(f"PREDICTION REPORT FOR: {target_date.date()}")
    print("="*50)
    print(f"Festival Name     : {fest_name if pd.notna(fest_name) else 'None (Ordinary or Window Day)'}")
    print(f"Is Festival Day?  : {'Yes (1)' if is_fest == 1 else 'No (0)'}")
    print(f"Relative Day Index: {rel_d if pd.notna(rel_d) else 'Outside Window'}")
    print(f"Predicted Price   : ₹{predicted_price:.2f}")
    print(f"Actual Price      : ₹{actual_price:.2f}")
    
    if predicted_price >= threshold_price:
        print(f"🚨 SPIKE ALERT: High price surge expected! (Above threshold ₹{threshold_price})")
    else:
        print(f"🌿 NORMAL STATUS: Standard pricing expected.")


# You can change this date to test any day in your dataset (e.g., a known festival date)
sample_date = "2016-1-15"  
predict_flower_price_for_date(sample_date)

MODEL ACCURACY EVALUATION (TEST SET)
R² Score Accuracy : 68.74% (Variance explained)
RMSE (Root Mean Sq) : ₹132.75 (Average error magnitude)
MAE (Mean Absolute) : ₹99.08 (Average absolute miss)

PREDICTION REPORT FOR: 2016-01-15
Festival Name     : Makara Sankranti
Is Festival Day?  : Yes (1)
Relative Day Index: 0.0
Predicted Price   : ₹585.99
Actual Price      : ₹720.00
🌿 NORMAL STATUS: Standard pricing expected.


In [4]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# =========================================================================
# 1. LOAD AND PREPARE DATASET
# =========================================================================
CSV_PATH = r'C:\Project\data\processed\flower_prices_with_rel_day.csv'
df = pd.read_csv(CSV_PATH)
df['DATE'] = pd.to_datetime(df['DATE'])
df = df.sort_values('DATE').reset_index(drop=True)

# 2. Extract basic calendar features (Seasonality)
df['month'] = df['DATE'].dt.month
df['day_of_week'] = df['DATE'].dt.dayofweek

# 3. Create Lag Features (Short-term momentum)
for lag in [1, 2, 3, 7]:
    df[f'price_lag_{lag}'] = df['price_updated'].shift(lag).bfill()

# 4. Fill dummy columns for the +/- 7 day festival countdown window
dummy_cols = [f'rel_day_{i}' if i < 0 else (f'rel_day_p{i}' if i > 0 else 'rel_day_0') for i in range(-7, 8)] + ['in_festival_window']
for col in dummy_cols:
    df[col] = df[col].fillna(0)

feature_cols = [
    'price_lag_1', 'price_lag_2', 'price_lag_3', 'price_lag_7',
    'is_festival', 'month', 'day_of_week'
] + dummy_cols

X = df[feature_cols]
y = df['price_updated']

# =========================================================================
# 5. CHRONOLOGICAL SPLIT
# =========================================================================
total_rows = len(df)
train_end = int(total_rows * 0.70)
val_end = int(total_rows * 0.85)

X_train, y_train = X.iloc[:train_end], y.iloc[:train_end]
X_test, y_test = X.iloc[val_end:], y.iloc[val_end:]

# =========================================================================
# 6. TRAIN LINEAR REGRESSION MODEL
# =========================================================================
linear_model = Pipeline([
    ('scaler', StandardScaler()),
    ('regressor', LinearRegression())
])
linear_model.fit(X_train, y_train)

# =========================================================================
# 7. EVALUATE MODEL ACCURACY (TEST SET)
# =========================================================================
y_pred = linear_model.predict(X_test)

r2 = r2_score(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae = mean_absolute_error(y_test, y_pred)

print("="*50)
print("LINEAR REGRESSION ACCURACY EVALUATION (TEST SET)")
print("="*50)
print(f"R² Score Accuracy : {r2 * 100:.2f}% (Variance explained)")
print(f"RMSE (Root Mean Sq) : ₹{rmse:.2f} (Average error magnitude)")
print(f"MAE (Mean Absolute) : ₹{mae:.2f} (Average absolute miss)")

# =========================================================================
# 8. TEST WITH TARGET DATE
# =========================================================================
def predict_linear(target_date_str, threshold_price=600):
    target_date = pd.to_datetime(target_date_str)
    match = df[df['DATE'] == target_date]
    if match.empty: return "Date not found."
    row_idx = match.index[0]
    
    pred = linear_model.predict(X.iloc[[row_idx]])[0]
    actual = y.iloc[row_idx]
    
    print(f"\nLinear Regression Report for {target_date.date()}:")
    print(f"Predicted: ₹{pred:.2f} | Actual: ₹{actual:.2f}")
    if pred >= threshold_price: print("🚨 SPIKE ALERT!")

predict_linear("2016-01-15")

LINEAR REGRESSION ACCURACY EVALUATION (TEST SET)
R² Score Accuracy : 68.79% (Variance explained)
RMSE (Root Mean Sq) : ₹132.65 (Average error magnitude)
MAE (Mean Absolute) : ₹98.88 (Average absolute miss)

Linear Regression Report for 2016-01-15:
Predicted: ₹573.45 | Actual: ₹720.00


In [5]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LassoCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# =========================================================================
# 1. LOAD AND PREPARE DATASET (Same Structure)
# =========================================================================
CSV_PATH = r'C:\Project\data\processed\flower_prices_with_rel_day.csv'
df = pd.read_csv(CSV_PATH)
df['DATE'] = pd.to_datetime(df['DATE'])
df = df.sort_values('DATE').reset_index(drop=True)

df['month'] = df['DATE'].dt.month
df['day_of_week'] = df['DATE'].dt.dayofweek

for lag in [1, 2, 3, 7]:
    df[f'price_lag_{lag}'] = df['price_updated'].shift(lag).bfill()

dummy_cols = [f'rel_day_{i}' if i < 0 else (f'rel_day_p{i}' if i > 0 else 'rel_day_0') for i in range(-7, 8)] + ['in_festival_window']
for col in dummy_cols:
    df[col] = df[col].fillna(0)

feature_cols = [
    'price_lag_1', 'price_lag_2', 'price_lag_3', 'price_lag_7',
    'is_festival', 'month', 'day_of_week'
] + dummy_cols

X = df[feature_cols]
y = df['price_updated']

# =========================================================================
# 2. CHRONOLOGICAL SPLIT
# =========================================================================
total_rows = len(df)
train_end = int(total_rows * 0.70)
val_end = int(total_rows * 0.85)

X_train, y_train = X.iloc[:train_end], y.iloc[:train_end]
X_test, y_test = X.iloc[val_end:], y.iloc[val_end:]

# =========================================================================
# 3. TRAIN LASSOCV MODEL (Finds best alpha automatically)
# =========================================================================
lassocv_model = Pipeline([
    ('scaler', StandardScaler()),
    ('regressor', LassoCV(cv=5, random_state=42, max_iter=10000))
])
lassocv_model.fit(X_train, y_train)

# Print the optimal alpha selected by cross-validation
print(f"Optimal Alpha chosen by LassoCV: {lassocv_model.named_steps['regressor'].alpha_:.4f}")

# =========================================================================
# 4. EVALUATE MODEL ACCURACY (TEST SET)
# =========================================================================
y_pred = lassocv_model.predict(X_test)

r2 = r2_score(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae = mean_absolute_error(y_test, y_pred)

print("="*50)
print("LASSOCV ACCURACY EVALUATION (TEST SET)")
print("="*50)
print(f"R² Score Accuracy : {r2 * 100:.2f}% (Variance explained)")
print(f"RMSE (Root Mean Sq) : ₹{rmse:.2f} (Average error magnitude)")
print(f"MAE (Mean Absolute) : ₹{mae:.2f} (Average absolute miss)")

# =========================================================================
# 5. TEST WITH TARGET DATE
# =========================================================================
def predict_lassocv(target_date_str, threshold_price=600):
    target_date = pd.to_datetime(target_date_str)
    match = df[df['DATE'] == target_date]
    if match.empty: return "Date not found."
    row_idx = match.index[0]
    
    pred = lassocv_model.predict(X.iloc[[row_idx]])[0]
    actual = y.iloc[row_idx]
    
    print(f"\nLassoCV Report for {target_date.date()}:")
    print(f"Predicted: ₹{pred:.2f} | Actual: ₹{actual:.2f}")
    if pred >= threshold_price: print("🚨 SPIKE ALERT!")

predict_lassocv("2016-01-15")

Optimal Alpha chosen by LassoCV: 1.9929
LASSOCV ACCURACY EVALUATION (TEST SET)
R² Score Accuracy : 68.74% (Variance explained)
RMSE (Root Mean Sq) : ₹132.75 (Average error magnitude)
MAE (Mean Absolute) : ₹99.85 (Average absolute miss)

LassoCV Report for 2016-01-15:
Predicted: ₹568.48 | Actual: ₹720.00


In [6]:
import pandas as pd
import numpy as np
from sklearn.linear_model import RidgeCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# =========================================================================
# 1. LOAD AND PREPARE DATASET (Same Structure)
# =========================================================================
CSV_PATH = r'C:\Project\data\processed\flower_prices_with_rel_day.csv'
df = pd.read_csv(CSV_PATH)
df['DATE'] = pd.to_datetime(df['DATE'])
df = df.sort_values('DATE').reset_index(drop=True)

df['month'] = df['DATE'].dt.month
df['day_of_week'] = df['DATE'].dt.dayofweek

for lag in [1, 2, 3, 7]:
    df[f'price_lag_{lag}'] = df['price_updated'].shift(lag).bfill()

dummy_cols = [f'rel_day_{i}' if i < 0 else (f'rel_day_p{i}' if i > 0 else 'rel_day_0') for i in range(-7, 8)] + ['in_festival_window']
for col in dummy_cols:
    df[col] = df[col].fillna(0)

feature_cols = [
    'price_lag_1', 'price_lag_2', 'price_lag_3', 'price_lag_7',
    'is_festival', 'month', 'day_of_week'
] + dummy_cols

X = df[feature_cols]
y = df['price_updated']

# =========================================================================
# 2. CHRONOLOGICAL SPLIT
# =========================================================================
total_rows = len(df)
train_end = int(total_rows * 0.70)
val_end = int(total_rows * 0.85)

X_train, y_train = X.iloc[:train_end], y.iloc[:train_end]
X_test, y_test = X.iloc[val_end:], y.iloc[val_end:]

# =========================================================================
# 3. TRAIN RIDGECV MODEL (Finds best alpha automatically)
# =========================================================================
# We provide a grid of alpha search values
alphas = [0.1, 1.0, 10.0, 100.0, 1000.0]
ridgecv_model = Pipeline([
    ('scaler', StandardScaler()),
    ('regressor', RidgeCV(alphas=alphas, cv=5))
])
ridgecv_model.fit(X_train, y_train)

print(f"Optimal Alpha chosen by RidgeCV: {ridgecv_model.named_steps['regressor'].alpha_}")

# =========================================================================
# 4. EVALUATE MODEL ACCURACY (TEST SET)
# =========================================================================
y_pred = ridgecv_model.predict(X_test)

r2 = r2_score(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae = mean_absolute_error(y_test, y_pred)

print("="*50)
print("RIDGECV ACCURACY EVALUATION (TEST SET)")
print("="*50)
print(f"R² Score Accuracy : {r2 * 100:.2f}% (Variance explained)")
print(f"RMSE (Root Mean Sq) : ₹{rmse:.2f} (Average error magnitude)")
print(f"MAE (Mean Absolute) : ₹{mae:.2f} (Average absolute miss)")

# =========================================================================
# 5. TEST WITH TARGET DATE
# =========================================================================
def predict_ridgecv(target_date_str, threshold_price=600):
    target_date = pd.to_datetime(target_date_str)
    match = df[df['DATE'] == target_date]
    if match.empty: return "Date not found."
    row_idx = match.index[0]
    
    pred = ridgecv_model.predict(X.iloc[[row_idx]])[0]
    actual = y.iloc[row_idx]
    
    print(f"\nRidgeCV Report for {target_date.date()}:")
    print(f"Predicted: ₹{pred:.2f} | Actual: ₹{actual:.2f}")
    if pred >= threshold_price: print("🚨 SPIKE ALERT!")

predict_ridgecv("2016-01-15")

Optimal Alpha chosen by RidgeCV: 0.1
RIDGECV ACCURACY EVALUATION (TEST SET)
R² Score Accuracy : 68.79% (Variance explained)
RMSE (Root Mean Sq) : ₹132.65 (Average error magnitude)
MAE (Mean Absolute) : ₹98.88 (Average absolute miss)

RidgeCV Report for 2016-01-15:
Predicted: ₹573.43 | Actual: ₹720.00
